In [54]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [55]:
from langchain_community.retrievers import WikipediaRetriever

retriever=WikipediaRetriever(top_k_results=3, lang="en")

In [56]:
query = "the geopolitical history of india and pakistan from the perspective of a chinese"
results = retriever.invoke(query)

In [57]:
results

[Document(metadata={'title': 'India–Pakistan war of 1971', 'summary': "The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.\nThirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as th

In [58]:
for i, doc in enumerate(results):
    print(f"Document: {i+1}")
    print(f"Title: {doc.metadata['title']}")
    print(f"Content: {doc.page_content[:500]}...")  # Print the first 500 characters of the content
    print("\n" + "="*80 + "\n")

Document: 1
Title: India–Pakistan war of 1971
Content: The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for Ea...


Document: 2
Title: China–India relations
Content: China and India maintained peaceful relations for thousands of years, but their relationship has varied since the Chinese Communist Party (CCP)'s victory in the Chinese Civil War in 1949 and the annexation of Tibet by the People's Republic of China. The two nations have sought economic cooperation with each other, while frequent border disputes and economic nationalism in both countries

## Vector Store Retriever

In [59]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community sentence-transformers


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [60]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document


embedding=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7508.31it/s]


In [61]:
docs = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [62]:
vector_store=Chroma.from_documents(
    embedding=embedding,
    collection_name="retriever_demo",
    documents=docs
)

In [63]:
retriever=vector_store.as_retriever(search_kwargs={"k": 2})

In [64]:
query='what is chroma used for'
res=retriever.invoke(query)

In [65]:
for i, doc in enumerate(res):
    print(f"Document: {i+1}")
    print(f"Content: {doc.page_content}")
    print("\n" + "="*80 + "\n")

Document: 1
Content: Chroma is a vector database optimized for LLM-based search.


Document: 2
Content: Chroma is a vector database optimized for LLM-based search.




## MMR

In [66]:
from langchain_community.vectorstores import FAISS

In [67]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [68]:
vector_store=FAISS.from_documents(
    embedding=embedding,
    documents=docs
)

In [69]:
retriever=vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={"k": 3, "lambda_mult": 0.2})

In [70]:
query = "What is langchain?"
results = retriever.invoke(query)
results

[Document(id='b59728a3-fc14-4f80-a400-54b63d2d4878', metadata={}, page_content='LangChain supports Chroma, FAISS, Pinecone, and more.'),
 Document(id='05e4e8e6-7991-411e-ae25-f583dad05a82', metadata={}, page_content='Embeddings are vector representations of text.'),
 Document(id='4bc79118-f527-4625-8c54-4ede4094aa1f', metadata={}, page_content='MMR helps you get diverse results when doing similarity search.')]

In [71]:
for i, doc in enumerate(results):
    print(f"Document: {i+1}")
    print(f"Content: {doc.page_content}")
    print("\n" + "="*80 + "\n")

Document: 1
Content: LangChain supports Chroma, FAISS, Pinecone, and more.


Document: 2
Content: Embeddings are vector representations of text.


Document: 3
Content: MMR helps you get diverse results when doing similarity search.




## Multiquery Retriever

In [72]:
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import MultiQueryRetriever

In [73]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [74]:
vector_store=FAISS.from_documents(
    embedding=embedding,
    documents=all_docs
)

In [75]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_google_genai import GoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

True

In [76]:
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
    llm=ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id="Qwen/Qwen2.5-72B-Instruct"))
)

In [77]:
# Query
query = "How to improve energy levels and maintain balance in health?"

similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

# similarity_results
multiquery_results

[Document(id='0f49de85-fb73-4589-8c2b-1c0ea49ca5df', metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.'),
 Document(id='2128d781-588c-4fa7-ab18-b0c54be04a43', metadata={'source': 'H2'}, page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.'),
 Document(id='5ce78fa9-0382-4e51-8b0b-3f085098bdbb', metadata={'source': 'H1'}, page_content='Regular walking boosts heart health and can reduce symptoms of depression.'),
 Document(id='99e0e9c3-d3d1-4fb4-bf55-dbdf82b88c1d', metadata={'source': 'I1'}, page_content='The solar energy system in modern homes helps balance electricity demand.'),
 Document(id='398453af-527a-484f-b6aa-0992470119d6', metadata={'source': 'H4'}, page_content='Mindfulness and controlled breathing lower cortisol and improve mental clarity.'),
 Document(id='d9a2167e-3d22-426f-baa5-67e14256837c', metadata={'source': 'I3'}, page_content='Photosynthesis enables plants t

In [78]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 4 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 5 ---
Regular walking boosts heart health and can reduce symptoms of depression.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
The solar energy system in modern homes helps balance electri